In [1]:
# Libraries import karo
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'notebook_connected'

# Data load karo
ticker = yf.Ticker("RELIANCE.NS")
df = ticker.history(start="2020-01-01", end="2026-04-05")

# Clean karo
df.index = df.index.tz_localize(None)
df = df.drop(columns=["Dividends", "Stock Splits"])
df = df[df['Volume'] > 0]

print("Data loaded successfully! ✅")
print("Shape:", df.shape)

Data loaded successfully! ✅
Shape: (1547, 5)


In [2]:
# Moving Averages banao
df['MA_7']  = df['Close'].rolling(window=7).mean()   # 7 din ka average
df['MA_21'] = df['Close'].rolling(window=21).mean()  # 21 din ka average
df['MA_50'] = df['Close'].rolling(window=50).mean()  # 50 din ka average
df['MA_200']= df['Close'].rolling(window=200).mean() # 200 din ka average

print("Moving Averages created! ✅")
print(df[['Close', 'MA_7', 'MA_21', 'MA_50', 'MA_200']].tail(5))

Moving Averages created! ✅
                  Close         MA_7        MA_21        MA_50       MA_200
Date                                                                       
2026-03-25  1413.099976  1405.371443  1397.800008  1415.300005  1447.095197
2026-03-27  1348.099976  1398.300014  1393.957148  1412.598005  1446.646909
2026-03-30  1343.900024  1389.128592  1391.357149  1410.420005  1446.151225
2026-04-01  1369.199951  1386.900007  1389.566668  1408.628003  1445.833337
2026-04-02  1350.500000  1377.771432  1387.500000  1406.480002  1445.370156


💡 Trader wali Insight:

Close < MA_7 < MA_21 < MA_50 < MA_200 — matlab price apne saare averages se neeche hai — ye bearish signal hai — price abhi downtrend mein hai! Ye insight traders use karte hain! 😎

In [4]:
# Moving Averages chart
fig = go.Figure()

fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Close Price', line=dict(color='white', width=1)))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_7'], name='MA 7', line=dict(color='#FF6B6B', width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_21'], name='MA 21', line=dict(color='#4ECDC4', width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_50'], name='MA 50', line=dict(color='#FFE66D', width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_200'], name='MA 200', line=dict(color='#A29BFE', width=2)))

fig.update_layout(
    title='Reliance Industries - Price with Moving Averages',
    xaxis_title='Date',
    yaxis_title='Price (₹)',
    template='plotly_dark',
    height=500,
    hovermode='x unified'
)

fig.show()

💡 Pro Insight:

Jab MA 7 MA 200 ko cross kare upar se — isse "Golden Cross" kehte hain — ye strong buy signal hota hai!
Jab MA 7 MA 200 ko cross kare neeche se — isse "Death Cross" kehte hain — ye sell signal hota hai!
Ye concepts ML model ke features mein bahut kaam aayenge! 😎

In [5]:
# RSI calculate karo
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['Close'])

print("RSI created! ✅")
print(df[['Close', 'RSI']].tail(5))

RSI created! ✅
                  Close        RSI
Date                              
2026-03-25  1413.099976  56.806425
2026-03-27  1348.099976  37.326751
2026-03-30  1343.900024  30.809765
2026-04-01  1369.199951  40.950609
2026-04-02  1350.500000  40.931929


RSI created! ✅
                  Close        RSI
Date                              
2026-03-25  1413.099976  56.806425
2026-03-27  1348.099976  37.326751
2026-03-30  1343.900024  30.809765
2026-04-01  1369.199951  40.950609
2026-04-02  1350.500000  40.931929


💡 Business Insight #8:

30 March 2026 ko RSI 30.8 — matlab stock almost oversold zone mein tha! Ye traders ke liye buying opportunity ka signal hota hai! Aur actually dekho — 01 April ko price ₹1343 se ₹1369 ho gayi! RSI ne sahi predict kiya! 😎

In [6]:
# RSI Chart
fig_rsi = go.Figure()

fig_rsi.add_trace(go.Scatter(
    x=df.index, y=df['RSI'],
    name='RSI',
    line=dict(color='#FF6B6B', width=1.5)
))

# Overbought line
fig_rsi.add_hline(y=70, line_dash="dash", 
                  line_color="#ef5350",
                  annotation_text="Overbought (70)")

# Oversold line  
fig_rsi.add_hline(y=30, line_dash="dash",
                  line_color="#26a69a",
                  annotation_text="Oversold (30)")

fig_rsi.update_layout(
    title='Reliance Industries - RSI (2020-2026)',
    xaxis_title='Date',
    yaxis_title='RSI Value',
    template='plotly_dark',
    height=400
)

fig_rsi.show()

💡 Business Insight #9:

2025 mein RSI 99 — ye almost impossible level hai! Iska matlab us waqt bahut zyada buying ho rahi thi — aur uske turant baad price crash hui! RSI ne perfectly predict kiya! 😎

In [8]:
# MACD calculate karo
exp12 = df['Close'].ewm(span=12, adjust=False).mean()
exp26 = df['Close'].ewm(span=26, adjust=False).mean()

df['MACD'] = exp12 - exp26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Histogram'] = df['MACD'] - df['MACD_Signal']

print("MACD created! ✅")
print(df[['Close', 'MACD', 'MACD_Signal', 'MACD_Histogram']].tail(5))

MACD created! ✅
                  Close       MACD  MACD_Signal  MACD_Histogram
Date                                                           
2026-03-25  1413.099976  -4.259190    -8.573069        4.313879
2026-03-27  1348.099976  -8.399951    -8.538446        0.138495
2026-03-30  1343.900024 -11.883450    -9.207446       -2.676003
2026-04-01  1369.199951 -12.459035    -9.857764       -2.601271
2026-04-02  1350.500000 -14.259743   -10.738160       -3.521583


In [11]:
# MACD Chart
fig_macd = make_subplots(rows=2, cols=1, 
                          shared_xaxes=True,
                          row_heights=[0.6, 0.4])

# Price line
fig_macd.add_trace(go.Scatter(
    x=df.index, y=df['Close'],
    name='Close Price',
    line=dict(color='white', width=1.5)
), row=1, col=1)

# MACD line
fig_macd.add_trace(go.Scatter(
    x=df.index, y=df['MACD'],
    name='MACD',
    line=dict(color='#4ECDC4', width=1.5)
), row=2, col=1)

# Signal line
fig_macd.add_trace(go.Scatter(
    x=df.index, y=df['MACD_Signal'],
    name='Signal',
    line=dict(color='#FF6B6B', width=1.5)
), row=2, col=1)

# Histogram
fig_macd.add_trace(go.Bar(
    x=df.index,
    y=df['MACD_Histogram'],
    name='Histogram',
    marker_color=df['MACD_Histogram'].apply(
        lambda x: '#26a69a' if x >= 0 else '#ef5350'
    )
), row=2, col=1)

fig_macd.update_layout(
    title='Reliance Industries - MACD (2020-2026)',
    template='plotly_dark',
    height=600,
    hovermode='x unified'
)

fig_macd.show()

In [10]:
from plotly.subplots import make_subplots
print("make_subplots imported! ✅")

make_subplots imported! ✅


In [12]:
# Bollinger Bands calculate karo
df['BB_Middle'] = df['Close'].rolling(window=20).mean()
df['BB_Upper'] = df['BB_Middle'] + (df['Close'].rolling(window=20).std() * 2)
df['BB_Lower'] = df['BB_Middle'] - (df['Close'].rolling(window=20).std() * 2)

print("Bollinger Bands created! ✅")
print(df[['Close', 'BB_Upper', 'BB_Middle', 'BB_Lower']].tail(5))

Bollinger Bands created! ✅
                  Close     BB_Upper    BB_Middle     BB_Lower
Date                                                          
2026-03-25  1413.099976  1434.133862  1396.250006  1358.366150
2026-03-27  1348.099976  1437.267021  1393.730005  1350.192989
2026-03-30  1343.900024  1438.964798  1390.585004  1342.205210
2026-04-01  1369.199951  1438.626230  1389.350000  1340.073770
2026-04-02  1350.500000  1439.357506  1388.975000  1338.592494


💡 Business Insight #10:

Close ₹1350 — BB_Lower ₹1338 — matlab price lower band ke bahut paas hai! Ye oversold signal hai — price bounce back kar sakti hai! 😎



In [13]:
# Bollinger Bands Chart
fig_bb = go.Figure()

# Upper band
fig_bb.add_trace(go.Scatter(
    x=df.index, y=df['BB_Upper'],
    name='Upper Band',
    line=dict(color='#FF6B6B', width=1, dash='dash')
))

# Middle band
fig_bb.add_trace(go.Scatter(
    x=df.index, y=df['BB_Middle'],
    name='Middle Band (MA20)',
    line=dict(color='#FFE66D', width=1.5)
))

# Lower band - fill between upper and lower
fig_bb.add_trace(go.Scatter(
    x=df.index, y=df['BB_Lower'],
    name='Lower Band',
    line=dict(color='#4ECDC4', width=1, dash='dash'),
    fill='tonexty',
    fillcolor='rgba(100, 100, 255, 0.1)'
))

# Price line
fig_bb.add_trace(go.Scatter(
    x=df.index, y=df['Close'],
    name='Close Price',
    line=dict(color='white', width=1.5)
))

fig_bb.update_layout(
    title='Reliance Industries - Bollinger Bands (2020-2026)',
    xaxis_title='Date',
    yaxis_title='Price (₹)',
    template='plotly_dark',
    height=500,
    hovermode='x unified'
)

fig_bb.show()

In [14]:
# Lag Features — kal ki price aaj ki prediction mein
df['Close_Lag1'] = df['Close'].shift(1)  # 1 din pehle ki price
df['Close_Lag2'] = df['Close'].shift(2)  # 2 din pehle ki price
df['Close_Lag3'] = df['Close'].shift(3)  # 3 din pehle ki price
df['Close_Lag5'] = df['Close'].shift(5)  # 5 din pehle ki price

# Date Features
df['Year']        = df.index.year
df['Month']       = df.index.month
df['Day']         = df.index.day
df['Day_of_Week'] = df.index.dayofweek  # 0=Monday, 4=Friday
df['Quarter']     = df.index.quarter

# Daily Return
df['Daily_Return'] = df['Close'].pct_change() * 100

print("Lag + Date Features created! ✅")
print("\nTotal columns ab:", df.shape[1])
print("\nSaare columns:")
print(df.columns.tolist())

Lag + Date Features created! ✅

Total columns ab: 26

Saare columns:
['Open', 'High', 'Low', 'Close', 'Volume', 'MA_7', 'MA_21', 'MA_50', 'MA_200', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Close_Lag1', 'Close_Lag2', 'Close_Lag3', 'Close_Lag5', 'Year', 'Month', 'Day', 'Day_of_Week', 'Quarter', 'Daily_Return']


In [17]:
# Missing values check karo — features banane ke baad
print("Missing values:")
print(df.isnull().sum())

Missing values:
Open                0
High                0
Low                 0
Close               0
Volume              0
MA_7                6
MA_21              20
MA_50              49
MA_200            199
RSI                13
MACD                0
MACD_Signal         0
MACD_Histogram      0
BB_Middle          19
BB_Upper           19
BB_Lower           19
Close_Lag1          1
Close_Lag2          2
Close_Lag3          3
Close_Lag5          5
Year                0
Month               0
Day                 0
Day_of_Week         0
Quarter             0
Daily_Return        1
dtype: int64


In [18]:
# Missing values wali rows drop karo
df = df.dropna()

print("Missing values hatane ke baad:")
print("Shape:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())

Missing values hatane ke baad:
Shape: (1348, 26)
Missing values remaining: 0


In [27]:
import os

# Folder banao agar exist nahi karta
os.makedirs(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\data', exist_ok=True)

# Ab save karo
df.to_csv(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\data\reliance_featured_data.csv')
print("Data saved! ✅")

Data saved! ✅


# Day 5 - Feature Engineering

## Libraries Used
- Pandas, NumPy, Plotly

## Features Created

### 1. Moving Averages (4 features)
- MA_7  → 7 din ka average price
- MA_21 → 21 din ka average price
- MA_50 → 50 din ka average price
- MA_200→ 200 din ka average price

### 2. RSI - Relative Strength Index (1 feature)
- 14 din ka RSI calculate kiya
- RSI > 70 = Overbought, RSI < 30 = Oversold

### 3. MACD (3 features)
- MACD = 12 EMA - 26 EMA
- MACD_Signal = 9 din ka EMA
- MACD_Histogram = MACD - Signal

### 4. Bollinger Bands (3 features)
- BB_Upper → Middle + 2 * Standard Deviation
- BB_Middle → 20 din ka Moving Average
- BB_Lower → Middle - 2 * Standard Deviation

### 5. Lag Features (4 features)
- Close_Lag1 → 1 din pehle ki price
- Close_Lag2 → 2 din pehle ki price
- Close_Lag3 → 3 din pehle ki price
- Close_Lag5 → 5 din pehle ki price

### 6. Date Features (5 features)
- Year, Month, Day, Day_of_Week, Quarter

### 7. Daily Return (1 feature)
- Har din ki % price change

## Final Dataset
- Total Rows: 1348
- Total Columns: 26
- Missing Values: 0
- Data saved: reliance_featured_data.csv

## Key Insights
- 5 raw columns se 26 features banaye
- MA_200 ki wajah se 199 rows drop hui — expected behavior
- Bollinger Bands se pata chala price lower band ke paas hai — oversold zone
- MACD negative hai — bearish momentum abhi
- RSI 40 ke aaspaas — neutral to oversold zone

In [24]:
# ATR - Average True Range
df['ATR'] = (df['High'] - df['Low']).rolling(window=14).mean()

# OBV - On Balance Volume
obv = [0]
for i in range(1, len(df)):
    if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
        obv.append(obv[-1] + df['Volume'].iloc[i])
    elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
        obv.append(obv[-1] - df['Volume'].iloc[i])
    else:
        obv.append(obv[-1])
df['OBV'] = obv

# Price Range aur Price Change
df['Price_Range'] = df['High'] - df['Low']
df['Price_Change'] = df['Close'] - df['Open']

# Missing values drop karo
df = df.dropna()

print("Extra features added! ✅")
print("Total columns:", df.shape[1])
print("Total rows:", df.shape[0])

Extra features added! ✅
Total columns: 30
Total rows: 1335


In [26]:
# Missing values final check
print("Missing values check:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())
print("Shape:", df.shape)

Missing values check:
Open              0
High              0
Low               0
Close             0
Volume            0
MA_7              0
MA_21             0
MA_50             0
MA_200            0
RSI               0
MACD              0
MACD_Signal       0
MACD_Histogram    0
BB_Middle         0
BB_Upper          0
BB_Lower          0
Close_Lag1        0
Close_Lag2        0
Close_Lag3        0
Close_Lag5        0
Year              0
Month             0
Day               0
Day_of_Week       0
Quarter           0
Daily_Return      0
ATR               0
OBV               0
Price_Range       0
Price_Change      0
dtype: int64

Total missing values: 0
Shape: (1335, 30)
